# 📁 Module 2.3: Logging Artifacts — Files, Plots & Datasets

**Goal**: Learn to log files as artifacts — plots, data snapshots, custom files, and more.

Artifacts are any **files** you want to associate with a run: visualizations, data samples, configuration files, etc.

---

In [ ]:
import mlflow
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    ConfusionMatrixDisplay
)
import json
import os

mlflow.set_experiment("02_logging_artifacts")

# Load data
wine = load_wine()
X_train, X_test, y_train, y_test = train_test_split(
    wine.data, wine.target, test_size=0.2, random_state=42
)
print("✅ Setup complete!")

## 1. Log a Single File — `log_artifact()`

The most basic way to log an artifact: save a file to disk, then log it.

In [ ]:
with mlflow.start_run(run_name="artifact_single_file"):
    
    # Train a model
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    mlflow.log_metric("accuracy", accuracy_score(y_test, y_pred))
    
    # --- Log a classification report as a text file ---
    report = classification_report(y_test, y_pred, target_names=wine.target_names)
    
    # Save to disk first
    with open("classification_report.txt", "w") as f:
        f.write(report)
    
    # Log the file as an artifact
    mlflow.log_artifact("classification_report.txt")
    
    # Clean up local file
    os.remove("classification_report.txt")
    
    print("✅ Classification report logged as artifact!")
    print(report)

## 2. Log to a Subdirectory — `artifact_path`

You can organize artifacts into subdirectories within the run.

In [ ]:
with mlflow.start_run(run_name="artifact_subdirectories"):
    
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    mlflow.log_metric("accuracy", accuracy_score(y_test, y_pred))
    
    # --- Log report to 'reports/' subdirectory ---
    report = classification_report(y_test, y_pred, target_names=wine.target_names)
    with open("classification_report.txt", "w") as f:
        f.write(report)
    mlflow.log_artifact("classification_report.txt", artifact_path="reports")
    os.remove("classification_report.txt")
    
    # --- Log config to 'config/' subdirectory ---
    config = {
        "model": "RandomForest",
        "n_estimators": 100,
        "dataset": "wine",
        "test_size": 0.2
    }
    with open("config.json", "w") as f:
        json.dump(config, f, indent=2)
    mlflow.log_artifact("config.json", artifact_path="config")
    os.remove("config.json")
    
    print("✅ Artifacts logged to subdirectories!")
    print("   reports/classification_report.txt")
    print("   config/config.json")

## 3. Log Matplotlib Plots

Visualizations are some of the most valuable artifacts. Let's log common ML plots.

In [ ]:
with mlflow.start_run(run_name="artifact_plots"):
    
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    accuracy = accuracy_score(y_test, y_pred)
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_params({"n_estimators": 100, "model": "RandomForest"})
    
    # --- Plot 1: Confusion Matrix ---
    fig, ax = plt.subplots(figsize=(8, 6))
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=wine.target_names)
    disp.plot(ax=ax, cmap='Blues')
    ax.set_title('Confusion Matrix')
    plt.tight_layout()
    fig.savefig("confusion_matrix.png", dpi=150)
    mlflow.log_artifact("confusion_matrix.png", artifact_path="plots")
    plt.show()
    os.remove("confusion_matrix.png")
    
    # --- Plot 2: Feature Importance ---
    fig, ax = plt.subplots(figsize=(10, 6))
    feature_importance = pd.Series(
        model.feature_importances_, 
        index=wine.feature_names
    ).sort_values(ascending=True)
    
    feature_importance.plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title('Feature Importance')
    ax.set_xlabel('Importance')
    plt.tight_layout()
    fig.savefig("feature_importance.png", dpi=150)
    mlflow.log_artifact("feature_importance.png", artifact_path="plots")
    plt.show()
    os.remove("feature_importance.png")
    
    print(f"\n✅ Plots logged! Accuracy: {accuracy:.4f}")

## 4. Log an Entire Directory — `log_artifacts()`

When you have **multiple related files**, you can log a whole directory at once.

In [ ]:
with mlflow.start_run(run_name="artifact_directory"):
    
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mlflow.log_metric("accuracy", accuracy_score(y_test, y_pred))
    
    # Create a temporary directory with multiple outputs
    os.makedirs("run_outputs", exist_ok=True)
    
    # Save multiple files
    # 1. Data sample
    df_sample = pd.DataFrame(X_test[:10], columns=wine.feature_names)
    df_sample['actual'] = y_test[:10]
    df_sample['predicted'] = y_pred[:10]
    df_sample.to_csv("run_outputs/predictions_sample.csv", index=False)
    
    # 2. Model config
    with open("run_outputs/config.json", "w") as f:
        json.dump({"model": "RandomForest", "n_estimators": 100}, f, indent=2)
    
    # 3. Classification report
    with open("run_outputs/report.txt", "w") as f:
        f.write(classification_report(y_test, y_pred, target_names=wine.target_names))
    
    # Log the entire directory!
    mlflow.log_artifacts("run_outputs", artifact_path="outputs")
    
    # Clean up
    import shutil
    shutil.rmtree("run_outputs")
    
    print("✅ Entire directory logged as artifacts!")
    print("   outputs/predictions_sample.csv")
    print("   outputs/config.json")
    print("   outputs/report.txt")

## 5. Log Text and JSON Directly (No File Needed)

MLFlow 2.x added helper methods to log text and JSON without creating temporary files.

In [ ]:
with mlflow.start_run(run_name="artifact_direct_log"):
    
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mlflow.log_metric("accuracy", accuracy_score(y_test, y_pred))
    
    # Log text directly — no need to create a file!
    mlflow.log_text(
        classification_report(y_test, y_pred, target_names=wine.target_names),
        "reports/classification_report.txt"
    )
    
    # Log JSON/dict directly
    mlflow.log_dict(
        {
            "model": "RandomForest",
            "n_estimators": 100,
            "accuracy": float(accuracy_score(y_test, y_pred)),
            "features_used": list(wine.feature_names),
        },
        "config/experiment_config.json"
    )
    
    print("✅ Text and JSON logged directly (no temp files needed)!")
    print("   Much cleaner than save → log → delete!")

## 🔑 Key Takeaways

| Method | Use For | Example |
|--------|---------|--------|
| `log_artifact(path)` | Single file | `log_artifact("plot.png")` |
| `log_artifact(path, artifact_path)` | File in subdirectory | `log_artifact("plot.png", "plots")` |
| `log_artifacts(dir_path)` | Entire directory | `log_artifacts("output_dir/")` |
| `log_text(text, filename)` | Text content directly | `log_text(report, "report.txt")` |
| `log_dict(dict, filename)` | JSON/dict directly | `log_dict(config, "config.json")` |

### Best Practices
1. **Organize with subdirectories** — use `artifact_path` to keep things tidy
2. **Always log visualizations** — plots are invaluable when reviewing experiments later
3. **Use `log_text()` and `log_dict()`** when possible — cleaner than file-based workflow
4. **Log data samples** — helps debug unexpected results

---

## ➡️ Next: `04_auto_logging.ipynb` — Let MLFlow do the logging for you!